In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [12]:
!pip install peft transformers datasets -q

import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

OPTIONS = ['A','B','C','D','E']
label_map = {'A':0,'B':1,'C':2,'D':3,'E':4}
id2opt   = {0:'A',1:'B',2:'C',3:'D',4:'E'}

train_ds = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')
test_ds  = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv',  split='train')

labels = [label_map[a] for a in train_ds['answer']]
print('Train size:', len(train_ds))
print('Test size:', len(test_ds))

Train size: 2000
Test size: 500


## Fine-tune DeBERTa (microsoft/deberta-v3-small)

In [13]:
DEBERTA_MODEL = 'microsoft/deberta-v3-small'
deb_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

def tokenize_deberta(example):
    # concatenate prompt + all options as one string
    text = str(example['prompt']) + ' ' + ' '.join([str(example[o]) for o in OPTIONS])
    enc  = deb_tokenizer(text, padding='max_length', truncation=True, max_length=256)
    enc['labels'] = label_map[example['answer']]
    return enc

deb_train = train_ds.map(tokenize_deberta, remove_columns=train_ds.column_names)
deb_train.set_format('torch')

deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL, num_labels=5, ignore_mismatched_sizes=True)

deb_args = TrainingArguments(
    output_dir='./deberta_ckpt',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_steps=100,
    save_strategy='epoch',
    logging_steps=50,
    report_to='none',
    fp16=False,
    bf16=False,          # ← turn off all mixed precision
)

deb_trainer = Trainer(model=deb_model, args=deb_args, train_dataset=deb_train)
deb_trainer.train()
deb_model.save_pretrained('./deberta_ckpt/final')
deb_tokenizer.save_pretrained('./deberta_ckpt/final')
print('DeBERTa fine-tuning done!')

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias       

Step,Training Loss
50,3.289976
100,3.496110
150,3.337769
200,3.233807
250,3.245178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa fine-tuning done!


## Fine-tune RoBERTa (roberta-base)

In [14]:
ROBERTA_MODEL = 'roberta-base'
rob_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

def tokenize_roberta(example):
    text = str(example['prompt']) + ' ' + ' '.join([str(example[o]) for o in OPTIONS])
    enc  = rob_tokenizer(text, padding='max_length', truncation=True, max_length=256)
    enc['labels'] = label_map[example['answer']]
    return enc

rob_train = train_ds.map(tokenize_roberta, remove_columns=train_ds.column_names)
rob_train.set_format('torch')

rob_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL, num_labels=5, ignore_mismatched_sizes=True)

rob_args = TrainingArguments(
    output_dir='./roberta_ckpt',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    save_strategy='epoch',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

rob_trainer = Trainer(model=rob_model, args=rob_args, train_dataset=rob_train)
rob_trainer.train()
rob_model.save_pretrained('./roberta_ckpt/final')
rob_tokenizer.save_pretrained('./roberta_ckpt/final')
print('RoBERTa fine-tuning done!')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/di

Step,Training Loss
50,3.216306
100,2.844218
150,1.675683
200,0.731663
250,0.298491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa fine-tuning done!


## Load fine-tuned models for inference

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

deb_model     = AutoModelForSequenceClassification.from_pretrained('./deberta_ckpt/final').to(device)
deb_tokenizer = AutoTokenizer.from_pretrained('./deberta_ckpt/final')
deb_model.eval()

rob_model     = AutoModelForSequenceClassification.from_pretrained('./roberta_ckpt/final').to(device)
rob_tokenizer = AutoTokenizer.from_pretrained('./roberta_ckpt/final')
rob_model.eval()

def get_probs(model, tokenizer, text):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=256, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1)[0].cpu().numpy()

def make_text(row):
    return str(row['prompt']) + ' ' + ' '.join([str(row[o]) for o in OPTIONS])

print('Models loaded!')

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Models loaded!


## Q1 — DeBERTa inference on row 25

In [26]:
row25      = test_ds[25]
text25     = make_text(row25)

deb_probs25 = get_probs(deb_model, deb_tokenizer, text25)
top_deb_idx = int(np.argmax(deb_probs25))

print('Q1 Answer:', id2opt[top_deb_idx], round(float(deb_probs25[top_deb_idx]), 4))

Q1 Answer: B 0.2406


## Q2 — Simple average ensemble on row 25

In [17]:
rob_probs25 = get_probs(rob_model, rob_tokenizer, text25)
avg_probs25 = (deb_probs25 + rob_probs25) / 2
top_avg_idx = int(np.argmax(avg_probs25))

print('Q2 Answer:', id2opt[top_avg_idx])

Q2 Answer: E


## Q3 — Weighted ensemble (DeBERTa 0.7, RoBERTa 0.3) on row 25

In [18]:
weighted_probs25 = 0.7 * deb_probs25 + 0.3 * rob_probs25
top_w_idx = int(np.argmax(weighted_probs25))

print('Q3 Answer:', id2opt[top_w_idx])

Q3 Answer: E


## Q4 — Top-3 prediction string for row 25

In [19]:
top3_idx = np.argsort(weighted_probs25)[::-1][:3]
top3_str = ' '.join([id2opt[i] for i in top3_idx])

print('Q4 Answer:', top3_str)

Q4 Answer: E B C


## Q5 — Run weighted ensemble on all test.csv, save submission.csv

In [20]:
rows = []
for row in test_ds:
    text = make_text(row)
    d_p  = get_probs(deb_model, deb_tokenizer, text)
    r_p  = get_probs(rob_model, rob_tokenizer, text)
    w_p  = 0.7 * d_p + 0.3 * r_p
    top3 = ' '.join([id2opt[i] for i in np.argsort(w_p)[::-1][:3]])
    rows.append({'id': row['id'], 'prediction': top3})

sub_df = pd.DataFrame(rows)
sub_df.to_csv('submission.csv', index=False)

print('Q5 Answer:', len(sub_df))

Q5 Answer: 500


## Q6 — TTA on first 50 test rows (DeBERTa only)

In [21]:
PREFIX = 'Answer the following multiple-choice question carefully: '

tta_diff = 0
for i in range(50):
    row  = test_ds[i]
    text_orig = make_text(row)
    text_aug  = PREFIX + text_orig

    p_orig = get_probs(deb_model, deb_tokenizer, text_orig)
    p_aug  = get_probs(deb_model, deb_tokenizer, text_aug)
    p_avg  = (p_orig + p_aug) / 2

    if np.argmax(p_orig) != np.argmax(p_avg):
        tta_diff += 1

print('Q6 Answer:', tta_diff)

Q6 Answer: 0


## Q7 — DeBERTa vs Weighted Ensemble Top-1 diff on first 100 test rows

In [22]:
q7_diff = 0
deb_top1_100  = []
ens_top1_100  = []
deb_conf_100  = []
ens_conf_100  = []
deb_top3_100  = []
ens_top3_100  = []

for i in range(100):
    row  = test_ds[i]
    text = make_text(row)

    d_p = get_probs(deb_model, deb_tokenizer, text)
    r_p = get_probs(rob_model, rob_tokenizer, text)
    w_p = 0.7 * d_p + 0.3 * r_p

    d_top1 = int(np.argmax(d_p))
    w_top1 = int(np.argmax(w_p))

    deb_top1_100.append(d_top1)
    ens_top1_100.append(w_top1)
    deb_conf_100.append(float(d_p[d_top1]))
    ens_conf_100.append(float(w_p[w_top1]))
    deb_top3_100.append([id2opt[j] for j in np.argsort(d_p)[::-1][:3]])
    ens_top3_100.append([id2opt[j] for j in np.argsort(w_p)[::-1][:3]])

    if d_top1 != w_top1:
        q7_diff += 1

print('Q7 Answer:', q7_diff)

Q7 Answer: 75


## Q8 — Positive confidence gain rows

In [23]:
gains = [ens_conf_100[i] - deb_conf_100[i] for i in range(100)]
q8 = sum(1 for g in gains if g > 0)
print('Q8 Answer:', q8)

Q8 Answer: 100


## Q9 — Rows with at least one change in Top-3 after ensembling

In [24]:
q9 = sum(1 for i in range(100) if deb_top3_100[i] != ens_top3_100[i])
print('Q9 Answer:', q9)

Q9 Answer: 75


## Q10 — MAP@3 on first 100 train rows using weighted ensemble

In [25]:
# Use train_ds for MAP@3 since we have ground truth labels
def ap_at_3(ranked, correct):
    for i, opt in enumerate(ranked):
        if opt == correct:
            return 1 / (i + 1)
    return 0.0

ap_scores = []
for i in range(100):
    row  = train_ds[i]
    text = make_text(row)
    d_p  = get_probs(deb_model, deb_tokenizer, text)
    r_p  = get_probs(rob_model, rob_tokenizer, text)
    w_p  = 0.7 * d_p + 0.3 * r_p
    top3 = [id2opt[j] for j in np.argsort(w_p)[::-1][:3]]
    ap_scores.append(ap_at_3(top3, row['answer']))

print('Q10 Answer:', round(float(np.mean(ap_scores)), 4))

Q10 Answer: 1.0
